# Przegląd surowych danych

Wybierz bazę, sesję i sygnały. Wykresy mają wspólną oś czasu: zaznaczenie zakresu przybliża wszystkie, dwuklik wraca do całości. Wykres sygnałów przybliża się tylko w czasie.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

from dms_analysis.overview import changing_pins, session_figure, sessions_table
from dms_analysis.raw import find_databases, load_session, load_sessions

DATA_DIR = Path("../data")
FIGURE_WIDTH = 1200

In [ ]:
databases = find_databases(DATA_DIR)
full_width = {"width": "100%"}

db_picker = widgets.Dropdown(
    options=[(str(path.relative_to(DATA_DIR)), path) for path in databases],
    description="Baza",
    layout={"width": "600px"},
)
session_picker = widgets.Dropdown(description="Sesja", layout={"width": "600px"})
pin_picker = widgets.HBox()
sessions_output = widgets.Output(layout={"width": "fit-content"})
figure_output = widgets.Output(layout=full_width)
session_data = None


def session_label(row):
    return f"{row.session_id}: {row.first_at:%Y-%m-%d %H:%M} – {row.last_at:%Y-%m-%d %H:%M} ({row.weight_count:.0f} wskazań)"


def selected_pins():
    return [box.pin for box in pin_picker.children if isinstance(box, widgets.Checkbox) and box.value]


def draw():
    figure = session_figure(session_data, f"{db_picker.label}, sesja {session_picker.value}", selected_pins(), FIGURE_WIDTH)
    with figure_output:
        figure_output.clear_output()
        display(figure)


def on_db_change(change=None):
    sessions = load_sessions(db_picker.value)
    with sessions_output:
        sessions_output.clear_output()
        display(sessions_table(sessions))
    with_data = sessions.dropna(subset=["first_at"])
    session_picker.options = [(session_label(row), row.session_id) for row in with_data.itertuples()]
    session_picker.value = with_data.loc[with_data["weight_count"].idxmax(), "session_id"]


def on_session_change(change=None):
    global session_data
    if session_picker.value is None:
        return
    session_data = load_session(db_picker.value, int(session_picker.value))
    changing = changing_pins(session_data.pins)
    boxes = []
    for pin in sorted(session_data.pins["pin"].unique()):
        box = widgets.Checkbox(value=pin in changing, description=f"pin {pin}", indent=False, layout={"width": "90px"})
        box.pin = int(pin)
        box.observe(lambda change: draw(), names="value")
        boxes.append(box)
    pin_picker.children = [widgets.Label("Sygnały:", layout={"width": "70px"}), *boxes]
    draw()


db_picker.observe(on_db_change, names="value")
session_picker.observe(on_session_change, names="value")
display(widgets.VBox([db_picker, sessions_output, session_picker, pin_picker, figure_output], layout=full_width))
on_db_change()